<a href="https://colab.research.google.com/github/pydeoxy/cic_ai_course/blob/main/langchain_llama3_2_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More about LangChain RAG:

https://learn.deeplearning.ai/courses/langchain-chat-with-your-data/

https://docs.langchain.com/oss/python/langchain/rag


## Dependencies

pip install transformers langchain langchain-huggingface langchain-community faiss-cpu

In [11]:
!pip install langchain langchain-huggingface langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.7 MB/s eta 0:00:00


In [4]:
# Import dependencies
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
    )
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import BeautifulSoupTransformer
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

In [15]:
# Turn on line wrapping
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
      white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)

## Load LLM and simple QA



In [5]:
# Load LLM
model_id = "unsloth/Llama-3.2-1B-Instruct"
# microsoft/Phi-4-mini-instruct
# google/gemma-3-1b-it
# meta-llama/Llama-3.2-1B-Instruct
# unsloth/Llama-3.2-1B-Instruct

# LLM
llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)


# Chat
chat = ChatHuggingFace(llm=llm, verbose=True)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [17]:
llm.invoke("What is SmartLab?")

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"What is SmartLab? SmartLab is a company that specializes in the development and manufacturing of smart home devices, particularly those related to home automation and security. Their products are designed to be easy to use, reliable, and secure, making them a popular choice among homeowners and businesses alike.\n\nSmartLab's products include a range of devices such as smart thermostats, security cameras, doorbells, and more. They also offer a range of accessories and integrations with popular smart home platforms like Amazon Alexa and Google Assistant.\n\nOne of the key features of SmartLab's products is their focus on ease of use and simplicity. They aim to make it easy for users to set up and manage their smart home devices without needing extensive technical knowledge or expertise.\n\nSmartLab's products are designed to be compatible with a wide range of devices and platforms, making it easy to integrate their products with other smart home systems. This allows users to create a s

In [18]:
messages = [
    ("system", "You are a helpful assistant."),
    ("human", "What is SmartLab?"),
]

chat.invoke(messages)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AIMessage(content="<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Apr 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is SmartLab?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSmartLab is a platform that allows users to create and manage their own virtual labs, where they can conduct experiments, simulations, and research in a safe and controlled environment. It's often used in educational settings, such as schools and universities, to teach various scientific concepts and skills.\n\nWith SmartLab, users can create virtual labs that mimic real-world experiments, allowing students to explore and learn about different scientific principles and phenomena in a hands-on way. The platform provides a range of features, including:\n\n1. Virtual lab environments: Users can create and customize their own virtual labs, including setting up equipment, adding materials, 

# RAG from webpage

In [8]:
# Load documents from webpage
urls = ["https://www.metropolia.fi/fi/tutkimus-kehitys-ja-innovaatiot/yhteistyoalustat/smart-lab"]
loader_html = AsyncHtmlLoader(urls)
docs_html = loader_html.load()

bs_transformer = BeautifulSoupTransformer()
docs_transformed = bs_transformer.transform_documents(docs_html, tags_to_extract=["div"])

# Split it into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs_transformed_split = text_splitter.split_documents(docs_transformed)


print(len(docs_transformed_split))
print(len(docs_transformed[0].page_content))

Fetching pages: 100%|##########| 1/1 [00:02<00:00,  2.08s/it]

19
16441


In [12]:
# create the open-source embedding function
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# load it into FAISS
html_db = FAISS.from_documents(docs_transformed_split, embedding_function)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrival QA

In [16]:
# Retriever
retriever_html = html_db.as_retriever()


# 1. Define the 'Answering' logic (The "Stuff" Chain)
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# This creates a chain that takes 'context' and 'input' and sends them to the LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 2. Define the 'Retrieval' logic
# This connects your vector store (retriever) to the answering chain
rag_chain = create_retrieval_chain(retriever_html, question_answer_chain)

# 3. Execution
# Note: The input key is now "input", and the answer is in "answer"
response = rag_chain.invoke({"input": "What is SmartLab?"})

print(response["answer"])

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.

kampuksella opiskellaan sosiaali- ja terveysalaa, rakennusalaa ja liiketaloutta.  Myyrmäen kampus (/fi/metropoliasta/kampukset/myyrmaki) Myyrmäen kampuksella Vantaalla opiskellaan tekniikkaa.   (/fi/verkkokauppa) Verkkokauppa  (/fi/verkkokauppa)     Etusivu (/fi)  Tutkimus-, kehitys- ja innovaatiotoiminta (/fi/tutkimus-kehitys-ja-innovaatiot)  Yhteistyöalustat (/fi/tutkimus-kehitys-ja-innovaatiot/yhteistyoalustat) Smart Lab - älykotien kehitysalusta       SmartLab – älykotien kehitysalusta Myllypuron kampuksella sijaitseva SmartLab on asumista palvelevan teknologian kehittämiseen ja testaukseen suunniteltu asunto, joka toimii myös koekenttänä innovaatioprojekteissa ja opinnäytetöissä. SmartLab on Skanskan (https://www.skanska.fi/) , ABB:n (https://new.abb.com/fi) ja Metropolian yhteinen hanke. Älyko

In [ ]:
# Clear RAM cache
#del pipe

# Clear CUDA cache
import torch
torch.cuda.empty_cache()